# Berrar P1 — Final TA-Approved Hybrid EPL Runner

This notebook runs the final P1 design:

- **StatsBomb EPL 2015/16** = target/modeling season and current-season causal history;
- **Football-Data EPL 2000/01–2014/15** = complete historical results used only for Berrar context;
- **Football-Data 2015/16 odds** = remains the separate market baseline in the existing MD1 pipeline.

The P1 historical parser admits only `Date`, `HomeTeam`, `AwayTeam`, `FTHG`, and `FTAG`. It hard-blocks incomplete seasons and searches the real Berrar recency range `n=9..100` using **training rows only**.


## 1. Configuration

Upload `md1_data_pipeline_code.zip` to `My Drive/ml_project/code/data_pipeline/`. The first real run downloads the 15 historical EPL CSV files and the StatsBomb EPL 2015/16 match metadata. Later runs reuse the Drive cache.


In [1]:
# Edit only these values if needed.
DRIVE_SUBFOLDER = "ml_project/code/data_pipeline"
CODE_ZIP_NAME = "md1_data_pipeline_code.zip"

RUN_SYNTHETIC_PREFLIGHT = True
P1_RUN_MODE = "real"                 # final run: "real"
P1_FULL_RECENCY_SEARCH = True         # final paper-aligned run: n=9..100
P1_RUN_KNN_SEARCH = False             # optional; substantially more expensive
P1_CUSTOM_RUN_NAME = None
OVERWRITE_DOWNLOADS = False
FORCE_REEXTRACT_CODE = False

# Optional: path to an already generated MD1 StatsBomb gold_prematch.parquet.
# If supplied, the notebook creates an Experiment-C augmented table on common matches.
BASELINE_GOLD_PREMATCH_PATH = None


## 2. Mount Drive and extract the package


In [2]:
import os, shutil, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_MYDRIVE = Path("/content/drive/MyDrive")
else:
    DRIVE_MYDRIVE = Path(os.environ.get("MD1_LOCAL_ROOT", str(Path.cwd()))).resolve()

DRIVE_WORKSPACE = DRIVE_MYDRIVE / DRIVE_SUBFOLDER
CODE_ZIP_PATH = DRIVE_WORKSPACE / CODE_ZIP_NAME
EXTRACT_ROOT = DRIVE_WORKSPACE / "extracted_p1_final"
RESULTS_ROOT = DRIVE_WORKSPACE / "results"
for p in (DRIVE_WORKSPACE, EXTRACT_ROOT, RESULTS_ROOT):
    p.mkdir(parents=True, exist_ok=True)
if not CODE_ZIP_PATH.exists():
    raise FileNotFoundError(f"Upload the implementation ZIP to: {CODE_ZIP_PATH}")

stamp = f"{CODE_ZIP_PATH.stat().st_size}:{CODE_ZIP_PATH.stat().st_mtime_ns}"
stamp_path = EXTRACT_ROOT / ".source_zip_stamp"
old = stamp_path.read_text().strip() if stamp_path.exists() else ""
if FORCE_REEXTRACT_CODE or stamp != old:
    shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP_PATH) as zf:
        zf.extractall(EXTRACT_ROOT)
    stamp_path.write_text(stamp)

PACKAGE_ROOT = EXTRACT_ROOT / "md1_data_pipeline_code"
if not PACKAGE_ROOT.exists():
    PACKAGE_ROOT = EXTRACT_ROOT
print("Package root:", PACKAGE_ROOT)
print("Results root:", RESULTS_ROOT)


Mounted at /content/drive
Package root: /content/drive/MyDrive/ml_project/code/data_pipeline/extracted_p1_final/md1_data_pipeline_code
Results root: /content/drive/MyDrive/ml_project/code/data_pipeline/results


## 3. Install dependencies, compile, and import


In [3]:
import subprocess, py_compile, importlib

req = PACKAGE_ROOT / "requirements_colab.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", "-r", str(req)])

for module_path in sorted((PACKAGE_ROOT / "md1_data_pipeline").glob("*.py")):
    py_compile.compile(str(module_path), doraise=True)

if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))
for module_name in list(sys.modules):
    if module_name == "md1_data_pipeline" or module_name.startswith("md1_data_pipeline."):
        del sys.modules[module_name]

from md1_data_pipeline import (
    PAPER_TOTAL_FEATURES, PAPER_HOMEAWAY_FEATURES,
    build_p1_default_config, execute_p1_workflow,
    merge_existing_prematch_with_p1,
)
print("Compilation and imports passed.")


ImportError: cannot import name 'PAPER_TOTAL_FEATURES' from 'md1_data_pipeline' (/content/drive/MyDrive/ml_project/code/data_pipeline/extracted_p1_final/md1_data_pipeline_code/md1_data_pipeline/__init__.py)

## 4. Optional synthetic preflight

This verifies source separation, cross-season continuity, gap continuity, train-only `n` selection, and the 6/18 feature contracts before downloading real data. The synthetic grid is `6..10`; the real run remains `9..100`.


In [ ]:
from IPython.display import display
import pandas as pd

if RUN_SYNTHETIC_PREFLIGHT:
    _, smoke_cfg = build_p1_default_config(results_root=RESULTS_ROOT, run_mode="synthetic_demo", run_name="P1_Hybrid_Synthetic_Preflight")
    smoke = execute_p1_workflow(config=smoke_cfg)
    print("Synthetic summary")
    display(pd.Series(smoke["summary"], name="value").to_frame())
    display(smoke["leakage_tests"][["test", "passed", "details"]])
    display(smoke["readiness"][["check", "passed", "details"]])
    assert smoke["readiness"]["passed"].all()
    print("Synthetic P1 preflight PASSED.")
else:
    print("Synthetic preflight skipped.")


## 5. Configure the real P1 run


In [ ]:
P1_RUN_NAME, p1_config = build_p1_default_config(
    results_root=RESULTS_ROOT,
    run_mode=P1_RUN_MODE,
    run_name=P1_CUSTOM_RUN_NAME,
    full_recency_search=P1_FULL_RECENCY_SEARCH,
)
p1_config.overwrite_downloads = OVERWRITE_DOWNLOADS
p1_config.p1_run_knn_search = P1_RUN_KNN_SEARCH

print("Run name:", P1_RUN_NAME)
print("Output root:", p1_config.project_root)
print("History provider:", p1_config.p1_history_provider)
print("Historical interval:", p1_config.p1_history_start_season, "through", p1_config.p1_history_end_season)
print("Target:", p1_config.competition_name, p1_config.season_name, "StatsBomb competition/season", p1_config.competition_id, p1_config.season_id)
print("Recency candidates:", p1_config.p1_recency_min, "..", p1_config.p1_recency_max)
print("TA approval note:", p1_config.p1_ta_approval_note)


## 6. Execute the real hybrid P1 pipeline

The run hard-blocks before results if any historical Football-Data season or the StatsBomb target season is incomplete.


In [ ]:
p1 = execute_p1_workflow(config=p1_config)
print(f"P1 completed in {p1['summary']['runtime_seconds']/60.0:.2f} minutes")
print("Readiness:", p1["summary"]["readiness_passed"])


## 7. Data-source and completeness evidence


In [ ]:
print("Source roles")
display(p1["source_roles"])

print("Historical EPL season completeness")
display(p1["history_season_audit"])

print("StatsBomb target-season completeness")
display(p1["target_season_audit"])

print("Cross-source team identity audit — target-related rows")
display(p1["team_identity"][p1["team_identity"]["is_target_season_team"]].head(50))


## 8. Berrar recency and feature outputs


In [ ]:
print("Run summary")
display(pd.Series(p1["summary"], name="value").to_frame())

print("Top Pearson candidates (training rows only)")
display(p1["pearson_audit"].sort_values("pearson_correlation", ascending=False).head(20))
print("Selected n =", p1["selected_n"])

print("Exact feature contracts")
print("total:", len([c for c in PAPER_TOTAL_FEATURES if c in p1["selected_total"].columns]))
print("homeaway:", len([c for c in PAPER_HOMEAWAY_FEATURES if c in p1["selected_homeaway"].columns]))

print("Historical coverage")
display(p1["history_coverage"])
print("Common comparison population")
display(p1["common_population"])


## 9. Leakage, source-contamination, readiness, and fidelity gates


In [ ]:
print("Leakage/source tests")
display(p1["leakage_tests"])

print("Readiness")
display(p1["readiness"])

print("Paper fidelity / project adaptations")
display(p1["fidelity"])

failed = p1["readiness"].loc[~p1["readiness"]["passed"]]
if not failed.empty:
    raise AssertionError("P1 readiness failed; results must not be used.")
print("ALL P1 READINESS CHECKS PASSED")


## 10. Optional Experiment-C merge with the existing MD1 pre-match table

Set `BASELINE_GOLD_PREMATCH_PATH` in Cell 1 if the verified StatsBomb MD1 `gold_prematch.parquet` already exists. This creates the common-match `MD1 + P1` representation without altering either source table.


In [ ]:
if BASELINE_GOLD_PREMATCH_PATH:
    baseline_path = Path(BASELINE_GOLD_PREMATCH_PATH)
    baseline = pd.read_parquet(baseline_path)
    augmented = merge_existing_prematch_with_p1(baseline, p1["selected_homeaway"], common_only=True)
    out_path = p1_config.gold_root / "md1_plus_p1_homeaway_common.parquet"
    augmented.to_parquet(out_path, index=False)
    print("Augmented common-population table:", out_path, "rows=", len(augmented))
else:
    print("BASELINE_GOLD_PREMATCH_PATH not set; Experiment-C merge skipped.")


## 11. Generated files


In [ ]:
print("P1 result root:", p1_config.project_root)
print("Selected total:", p1_config.gold_root / "p1_features_total_selected.parquet")
print("Selected homeaway:", p1_config.gold_root / "p1_features_homeaway_selected.parquet")
print("All n candidates:", p1_config.gold_root / "p1_features_total_all_n.parquet")
print("Pearson audit:", p1_config.audit_root / "p1_pearson_recency_search.csv")
print("Completeness audit:", p1_config.audit_root / "p1_season_completeness.csv")
print("Source roles:", p1_config.audit_root / "p1_source_roles.csv")
print("Leakage tests:", p1_config.audit_root / "p1_leakage_tests.csv")
print("Readiness:", p1_config.audit_root / "p1_readiness.csv")
print("Fidelity table:", p1_config.audit_root / "p1_fidelity_table.csv")
print("Run summary:", p1_config.audit_root / "p1_run_summary.json")
